In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))


In [ ]:
from src.chunker import *
from src.config import *
from src.data_loader import *
from src.evaluator import *
from src.generator import *
from src.retriever import *
from src.train_model import *
from src.utils import *

In [16]:
try:
    embeddings = HuggingFaceEmbeddings(
    model_name=TRAIN_MODEL_PATH,
    model_kwargs={'device': 'cuda'}
)
except :
    embeddings = None

In [ ]:
if embeddings is None:
    embeddings = train_model()

In [ ]:
loader = RCKDataLoader(embeddings=embeddings, db_index_name=DB_INDEX_NAME)

In [ ]:

db = loader.get_index_db(DB_PATH, force_rebuild=True)

In [ ]:
all_docs = list(db.docstore._dict.values())

In [ ]:
retriever = AdvancedHybridRetriever(db, all_docs)


In [ ]:
generator = EnhancedGenerator('Qwen/Qwen2.5-14B-Instruct')

In [ ]:
import pandas as pd


OUTPUT_EXCEL_PATH = OUTPUT_PATH / "dataset_with_rag_answers.xlsx"
NUMBER_RELEVANT_CHUNKS = 5


df = pd.read_excel(DATASET_EXCEL_PATH)


if 'question' not in df.columns:
    raise ValueError("В Excel-файле отсутствует столбец 'question'")


rag_answers = [] 
rag_context = []
eval_metrics = []
print(f"Всего вопросов для обработки: {len(df)}")

for idx, row in df.iterrows():
    question = row['question']
    print(f"\n[{idx+1}/{len(df)}] Вопрос: {question[:100]}...")  
    try:
    
        docs = retriever.retrieve(question, k=NUMBER_RELEVANT_CHUNKS)
        context_str = retriever.get_context_str(docs)
        answer = generator.generate(question, context_str)

        rag_answers.append(answer)
        rag_context.append(context_str)
    except Exception as e:
        print(f"   Ошибка при обработке вопроса: {e}")
        rag_answers.append("ERROR") 


df['RAG_answer'] = rag_answers
df['RAG_context'] = rag_context

df.to_excel(OUTPUT_EXCEL_PATH, index=False)
print(f"\nГотово! Результаты сохранены в файл: {OUTPUT_EXCEL_PATH}")

